# 01 · Synthetic Data Generation

**NYSED Data Integrity & Risk Monitoring Platform**
Office of Information & Reporting Services (IRS) — Special Education & Federal Data Collections

> ⚠️ **All data in this project is entirely synthetic.** District names, counts, submission
> records, and compliance figures are randomly generated for demonstration purposes only and do
> not represent any real school district, student, or New York State agency data.

This notebook generates a self-consistent synthetic dataset that mimics the shape of a state
education agency's data collection operation:

1. **Districts** — a synthetic roster of school districts across five NYS regions
2. **Submissions** — monthly collection submissions (SIRS, IDEA/618, EMSC, Personnel) with
   completeness, validity, timeliness, and duplicate metrics
3. **IDEA indicators** — State Performance Plan-style compliance indicators by district/year
4. **Risk scores** — a composite district-year risk determination (four tiers, OSEP-style)
5. **Validation rules & violations** — a rule catalog and simulated rule-violation counts
6. **Student roster** — a synthetic student roster with injected near-duplicate records, used
   later for record-linkage / duplicate-detection demonstrations

All downstream notebooks (`02`–`05`) read the CSVs written here from `../data/`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
import json

RNG = np.random.default_rng(207)   # fixed seed for reproducibility
DATA_DIR = Path("../data")
DOCS_DIR = Path("../docs")
ASSETS_DIR = DOCS_DIR / "assets"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print("Environment ready.")

Environment ready.


## 1. Regions, counties & synthetic districts

In [2]:
REGIONS = {
    "NYC Metro":      ["Kings", "Queens", "Bronx", "New York", "Richmond", "Nassau", "Westchester"],
    "Western NY":     ["Erie", "Monroe", "Niagara", "Chautauqua"],
    "Central NY":     ["Onondaga", "Oneida", "Broome", "Chemung"],
    "Capital Region": ["Albany", "Rensselaer", "Schenectady", "Saratoga"],
    "North Country":  ["Clinton", "Jefferson", "St. Lawrence", "Franklin"],
}

DISTRICTS_PER_REGION = 12

NAME_PREFIX = ["River", "Maple", "Cedar", "Lake", "North", "South", "East", "West", "Hill",
               "Stone", "Oak", "Pine", "Elm", "Birch", "Fox", "Silver", "Golden", "Union",
               "Liberty", "Harbor", "Ridge", "Valley", "Meadow", "Spring", "Franklin", "Highland"]
NAME_BASE = ["brook", "field", "ton", "view", "dale", "wood", "port", "ville", "haven",
             "crest", "mont", "burg", "glen", "side", "gate"]

def make_district_name(rng, size_tier):
    prefix = rng.choice(NAME_PREFIX)
    base = rng.choice(NAME_BASE)
    stem = f"{prefix}{base}"
    if size_tier == "Large City":
        return f"{stem.capitalize()} City SD"
    suffix = rng.choice(["CSD", "UFSD"], p=[0.65, 0.35])
    return f"{stem.capitalize()} {suffix}"

def draw_size_tier(rng):
    return rng.choice(
        ["Large City", "Suburban", "Small/Rural"],
        p=[0.08, 0.42, 0.50]
    )

ENROLL_RANGE = {
    "Large City":  (8000, 45000),
    "Suburban":    (1500, 8000),
    "Small/Rural": (200, 1500),
}

rows = []
did = 100000
used_names = set()
for region, counties in REGIONS.items():
    for i in range(DISTRICTS_PER_REGION):
        did += 1
        tier = draw_size_tier(RNG)
        # retry name generation to avoid duplicates
        for _ in range(20):
            name = make_district_name(RNG, tier)
            if name not in used_names:
                used_names.add(name)
                break
        county = RNG.choice(counties)
        lo, hi = ENROLL_RANGE[tier]
        # log-uniform draw: naturally occurring multi-order-of-magnitude count data
        # (like enrollment figures) approximates Benford's Law; a plain uniform draw does not.
        enrollment = int(round(10 ** RNG.uniform(np.log10(lo), np.log10(hi))))
        swd_pct = float(np.clip(RNG.normal(17.5, 3.2), 8, 32))
        swd_count = int(round(enrollment * swd_pct / 100))
        urbanicity = {"Large City": "Urban", "Suburban": "Suburban",
                      "Small/Rural": RNG.choice(["Rural", "Small Town"], p=[0.6, 0.4])}[tier]

        # Latent operational-quality parameter drives most downstream metrics.
        # Mixture: most districts perform well; a minority are chronic low performers.
        if RNG.random() < 0.13:
            quality = float(RNG.beta(2.5, 5.0))        # lower-performing tail
        else:
            quality = float(RNG.beta(11, 1.6))          # solid performers
        # Small/rural districts run slightly thinner data-ops staffing on average
        if tier == "Small/Rural":
            quality = float(np.clip(quality - RNG.uniform(0, 0.08), 0.02, 0.99))

        rows.append(dict(
            district_id=f"D{did}",
            district_name=name,
            region=region,
            county=county,
            size_tier=tier,
            urbanicity=urbanicity,
            enrollment_total=enrollment,
            swd_count=swd_count,
            swd_pct=round(swd_pct, 1),
            quality_latent=round(quality, 4),
        ))

districts = pd.DataFrame(rows)
print(f"Generated {len(districts)} synthetic districts across {len(REGIONS)} regions.")
districts.head()

Generated 60 synthetic districts across 5 regions.


,district_id,district_name,region,county,size_tier,urbanicity,enrollment_total,swd_count,swd_pct,quality_latent
0,D100001,Eastville UFSD,NYC Metro,Nassau,Suburban,Suburban,5832,1158,19.9,0.6956
1,D100002,Foxglen CSD,NYC Metro,New York,Small/Rural,Small Town,396,69,17.5,0.2613
2,D100003,Southton CSD,NYC Metro,Kings,Small/Rural,Small Town,695,120,17.2,0.8271
3,D100004,Maplefield CSD,NYC Metro,Queens,Suburban,Suburban,7277,1061,14.6,0.8380
4,D100005,Highlandhaven CSD,NYC Metro,Kings,Suburban,Suburban,2375,507,21.3,0.8976


In [3]:
districts.to_csv(DATA_DIR / "districts.csv", index=False)
districts[["region", "size_tier"]].value_counts().unstack(fill_value=0)

size_tier,Suburban,Small/Rural,Large City
region,,,
NYC Metro,6,6,0
Western NY,3,7,2
Central NY,5,3,4
Capital Region,5,6,1
North Country,6,6,0


## 2. Monthly collection submissions

Eight recurring collections, 24 reporting months (two school years), for every district.

In [4]:
COLLECTIONS = {
    "SIRS Enrollment":            dict(base=lambda d: d.enrollment_total, freq="monthly"),
    "SIRS Assessment":            dict(base=lambda d: int(d.enrollment_total * 0.62), freq="monthly"),
    "SIRS Special Education":     dict(base=lambda d: d.swd_count, freq="monthly"),
    "IDEA 618 Child Count":       dict(base=lambda d: d.swd_count, freq="monthly"),
    "IDEA 618 Exiting":           dict(base=lambda d: max(5, int(d.swd_count * 0.11)), freq="monthly"),
    "IDEA Maintenance of Effort": dict(base=lambda d: 1, freq="monthly"),
    "Student Attendance (EMSC)":  dict(base=lambda d: d.enrollment_total, freq="monthly"),
    "Personnel Master File":      dict(base=lambda d: max(10, int(d.enrollment_total * 0.07)), freq="monthly"),
}

# Two school years: Sep 2024 - Aug 2025, Sep 2025 - Aug 2026
months = pd.period_range("2024-09", "2026-08", freq="M")

def school_year_label(period):
    y = period.year if period.month >= 9 else period.year - 1
    return f"{y}-{str(y + 1)[-2:]}"

sub_rows = []
for _, d in districts.iterrows():
    q = d.quality_latent
    for coll_name, spec in COLLECTIONS.items():
        expected = max(1, int(spec["base"](d)))
        for m in months:
            # monthly noise around the latent quality trait (slow drift + noise)
            drift = RNG.normal(0, 0.03)
            month_quality = float(np.clip(q + drift, 0.02, 0.995))

            completeness_rate = float(np.clip(RNG.beta(month_quality * 45 + 3, (1 - month_quality) * 5 + 1), 0.55, 1.0))
            submitted = int(round(expected * completeness_rate))

            validity_error_rate = float(np.clip(RNG.beta((1 - month_quality) * 12 + 0.2, month_quality * 90 + 4), 0.0, 0.35))
            validity_error_count = int(round(submitted * validity_error_rate))

            duplicate_count = int(RNG.poisson(lam=max(0.05, (1 - month_quality) * 2.5)))

            on_time_prob = float(np.clip(0.60 + 0.40 * month_quality, 0.05, 0.99))
            on_time = bool(RNG.random() < on_time_prob)

            # accuracy audit only on a ~10% sample of district-collection-months
            audited = RNG.random() < 0.10
            if audited:
                accuracy_score = float(np.clip(RNG.beta(month_quality * 18 + 1, (1 - month_quality) * 4 + 1), 0.5, 1.0))
            else:
                accuracy_score = np.nan

            if validity_error_rate > 0.15 or completeness_rate < 0.75:
                status = "Rejected"
            elif validity_error_rate > 0.05 or completeness_rate < 0.92 or not on_time:
                status = "Accepted with Warnings"
            else:
                status = "Accepted"

            sub_rows.append(dict(
                district_id=d.district_id,
                region=d.region,
                collection=coll_name,
                period=str(m),
                school_year=school_year_label(m),
                records_expected=expected,
                records_submitted=submitted,
                completeness_rate=round(completeness_rate, 4),
                validity_error_count=validity_error_count,
                validity_error_rate=round(validity_error_rate, 4),
                duplicate_count=duplicate_count,
                on_time=on_time,
                accuracy_score=None if np.isnan(accuracy_score) else round(accuracy_score, 4),
                status=status,
            ))

submissions = pd.DataFrame(sub_rows)
print(f"Generated {len(submissions):,} submission records "
      f"({districts.shape[0]} districts x {len(COLLECTIONS)} collections x {len(months)} months).")
submissions.head()

Generated 11,520 submission records (60 districts x 8 collections x 24 months).


,district_id,region,collection,period,school_year,records_expected,records_submitted,completeness_rate,validity_error_count,validity_error_rate,duplicate_count,on_time,accuracy_score,status
0,D100001,NYC Metro,SIRS Enrollment,2024-09,2024-25,5832,5468,0.9375,64,0.0117,0,True,NaN,Accepted
1,D100001,NYC Metro,SIRS Enrollment,2024-10,2024-25,5832,5597,0.9598,518,0.0925,0,True,NaN,Accepted with Warnings
2,D100001,NYC Metro,SIRS Enrollment,2024-11,2024-25,5832,5707,0.9786,229,0.0401,0,True,NaN,Accepted
3,D100001,NYC Metro,SIRS Enrollment,2024-12,2024-25,5832,5385,0.9233,236,0.0439,1,True,NaN,Accepted
4,D100001,NYC Metro,SIRS Enrollment,2025-01,2024-25,5832,5140,0.8813,198,0.0385,0,True,NaN,Accepted with Warnings


In [5]:
submissions.to_csv(DATA_DIR / "submissions.csv", index=False)
submissions["status"].value_counts(normalize=True).round(3)

status
Accepted                  0.609
Accepted with Warnings    0.290
Rejected                  0.101
Name: proportion, dtype: float64

## 3. IDEA / State Performance Plan indicators

Six representative Part B indicators, by district and school year. Indicator direction (higher-is-better vs. lower-is-better) is tracked so downstream logic can evaluate 'met target' correctly.

In [6]:
INDICATORS = [
    dict(id="Ind-01", name="Graduation Rate (SWD)",               target=0.80, direction="higher", noise=0.10),
    dict(id="Ind-02", name="Dropout Rate (SWD)",                   target=0.06, direction="lower",  noise=0.04),
    dict(id="Ind-04", name="Suspension/Expulsion — Significant Discrepancy", target=0.0, direction="lower", noise=0.05, binary_risk=True),
    dict(id="Ind-09", name="Disproportionate Representation by Race/Ethnicity", target=0.0, direction="lower", noise=0.05, binary_risk=True),
    dict(id="Ind-11", name="Timely Initial Evaluation (60-day rule)", target=0.95, direction="higher", noise=0.08),
    dict(id="Ind-13", name="Secondary Transition IEP Compliance",    target=0.95, direction="higher", noise=0.09),
]

years = ["2024-25", "2025-26"]
# difficulty = the quality_latent value at which a district has ~50% odds of meeting the indicator
INDICATOR_DIFFICULTY = {"Ind-01": 0.42, "Ind-02": 0.38, "Ind-04": 0.52, "Ind-09": 0.52, "Ind-11": 0.58, "Ind-13": 0.56}

ind_rows = []
for _, d in districts.iterrows():
    q = d.quality_latent
    for yr in years:
        for ind in INDICATORS:
            threshold = INDICATOR_DIFFICULTY[ind["id"]]
            p_met = 1 / (1 + np.exp(-9 * (q - threshold)))
            met = bool(RNG.random() < p_met)

            if ind["direction"] == "higher":
                actual = ind["target"] + (RNG.uniform(0.0, 0.05) if met else -RNG.uniform(0.03, 0.25))
            else:
                actual = ind["target"] - (RNG.uniform(0.0, 0.02) if met else -RNG.uniform(0.02, 0.20))
            actual = float(np.clip(actual, 0, 1))

            # small districts => small denominators => noisier raw rates (motivates shrinkage later)
            if d.size_tier == "Small/Rural":
                denom = int(RNG.integers(5, 40))
            elif d.size_tier == "Suburban":
                denom = int(RNG.integers(30, 200))
            else:
                denom = int(RNG.integers(150, 900))

            ind_rows.append(dict(
                district_id=d.district_id,
                region=d.region,
                school_year=yr,
                indicator_id=ind["id"],
                indicator_name=ind["name"],
                direction=ind["direction"],
                target=ind["target"],
                actual_rate=round(actual, 4),
                denominator_n=denom,
                met_target=met,
            ))

idea_indicators = pd.DataFrame(ind_rows)
idea_indicators.to_csv(DATA_DIR / "idea_indicators.csv", index=False)
print(f"Generated {len(idea_indicators):,} indicator records.")
idea_indicators.head(8)

Generated 720 indicator records.


,district_id,region,school_year,indicator_id,indicator_name,direction,target,actual_rate,denominator_n,met_target
0,D100001,NYC Metro,2024-25,Ind-01,Graduation Rate (SWD),higher,0.80,0.8086,42,True
1,D100001,NYC Metro,2024-25,Ind-02,Dropout Rate (SWD),lower,0.06,0.0478,122,True
2,D100001,NYC Metro,2024-25,Ind-04,Suspension/Expulsion — Significant Discrepancy,lower,0.00,0.0000,87,True
3,D100001,NYC Metro,2024-25,Ind-09,Disproportionate Representation by Race/Ethnicity,lower,0.00,0.0000,151,True
4,D100001,NYC Metro,2024-25,Ind-11,Timely Initial Evaluation (60-day rule),higher,0.95,0.9588,49,True
5,D100001,NYC Metro,2024-25,Ind-13,Secondary Transition IEP Compliance,higher,0.95,0.9576,124,True
6,D100001,NYC Metro,2025-26,Ind-01,Graduation Rate (SWD),higher,0.80,0.8068,76,True
7,D100001,NYC Metro,2025-26,Ind-02,Dropout Rate (SWD),lower,0.06,0.0509,62,True


## 4. Composite district-year risk score & OSEP-style risk tier

Combines submission-quality metrics with indicator compliance into a single standardized composite, then buckets districts into four risk tiers using a realistic skewed distribution (most districts *Meets Requirements*, a small tail needing intervention).

In [7]:
def composite_for(district_id, yr):
    sub = submissions[(submissions.district_id == district_id) & (submissions.school_year == yr)]
    ind = idea_indicators[(idea_indicators.district_id == district_id) & (idea_indicators.school_year == yr)]
    return dict(
        district_id=district_id,
        school_year=yr,
        mean_completeness=sub.completeness_rate.mean(),
        mean_validity_error=sub.validity_error_rate.mean(),
        on_time_rate=sub.on_time.mean(),
        mean_duplicate_rate=(sub.duplicate_count / sub.records_submitted.replace(0, np.nan)).mean(),
        pct_indicators_met=ind.met_target.mean(),
    )

risk_rows = [composite_for(did_, yr) for did_ in districts.district_id for yr in years]
risk = pd.DataFrame(risk_rows)

def z(s):
    return (s - s.mean()) / s.std(ddof=0)

# Standardize; flip sign on "bad-when-high" metrics so higher z = worse
risk["z_completeness"]   = -z(risk.mean_completeness)
risk["z_validity_error"] =  z(risk.mean_validity_error)
risk["z_on_time"]        = -z(risk.on_time_rate)
risk["z_duplicate"]      =  z(risk.mean_duplicate_rate)
risk["z_indicators"]     = -z(risk.pct_indicators_met)

weights = dict(z_completeness=0.25, z_validity_error=0.25, z_on_time=0.15,
               z_duplicate=0.10, z_indicators=0.25)
risk["risk_score"] = sum(risk[c] * w for c, w in weights.items())

# Rescale to a friendlier 0-100 "risk index" (higher = riskier)
rmin, rmax = risk.risk_score.min(), risk.risk_score.max()
risk["risk_index"] = ((risk.risk_score - rmin) / (rmax - rmin) * 100).round(1)

q60, q85, q95 = risk.risk_index.quantile([0.60, 0.85, 0.95])
def tier(v):
    if v <= q60: return "Meets Requirements"
    if v <= q85: return "Needs Assistance"
    if v <= q95: return "Needs Intervention"
    return "Needs Substantial Intervention"

risk["risk_tier"] = risk.risk_index.apply(tier)
risk = risk.merge(districts[["district_id", "district_name", "region", "size_tier"]], on="district_id")

risk_scores = risk[["district_id", "district_name", "region", "size_tier", "school_year",
                     "mean_completeness", "mean_validity_error", "on_time_rate",
                     "mean_duplicate_rate", "pct_indicators_met", "risk_index", "risk_tier"]]
risk_scores.to_csv(DATA_DIR / "risk_scores.csv", index=False)
print(risk_scores.risk_tier.value_counts())
risk_scores.head()

risk_tier
Meets Requirements                72
Needs Assistance                  30
Needs Intervention                12
Needs Substantial Intervention     6
Name: count, dtype: int64


,district_id,district_name,region,size_tier,school_year,mean_completeness,mean_validity_error,on_time_rate,mean_duplicate_rate,pct_indicators_met,risk_index,risk_tier
0,D100001,Eastville UFSD,NYC Metro,Suburban,2024-25,0.922758,0.056688,0.875000,0.116093,1.000000,16.7,Needs Assistance
1,D100001,Eastville UFSD,NYC Metro,Suburban,2025-26,0.937440,0.052618,0.875000,0.105114,0.666667,21.8,Needs Assistance
2,D100002,Foxglen CSD,NYC Metro,Small/Rural,2024-25,0.765339,0.242599,0.750000,0.326780,0.000000,76.4,Needs Intervention
3,D100002,Foxglen CSD,NYC Metro,Small/Rural,2025-26,0.758078,0.244794,0.677083,0.268001,0.333333,71.2,Needs Intervention
4,D100003,Southton CSD,NYC Metro,Small/Rural,2024-25,0.951775,0.029768,0.958333,0.016467,1.000000,5.6,Meets Requirements


## 5. Validation rule catalog & violations

In [8]:
RULE_CATALOG = [
    dict(rule_id="R001", category="Completeness", severity="Error",   description="Required field missing"),
    dict(rule_id="R002", category="Range",        severity="Error",   description="Grade level outside valid range (PK-12)"),
    dict(rule_id="R003", category="Domain",       severity="Error",   description="Invalid disability classification code"),
    dict(rule_id="R004", category="Consistency",  severity="Error",   description="Exit date precedes entry date"),
    dict(rule_id="R005", category="Consistency",  severity="Warning", description="SWD flag set but no IEP date on file"),
    dict(rule_id="R006", category="Referential",  severity="Error",   description="District ID not found in master file"),
    dict(rule_id="R007", category="Referential",  severity="Warning", description="Student ID not found in enrollment extract"),
    dict(rule_id="R008", category="Uniqueness",   severity="Warning", description="Duplicate student record within district"),
    dict(rule_id="R009", category="Range",        severity="Error",   description="Date of birth implies age outside 3-21"),
    dict(rule_id="R010", category="Format",       severity="Error",   description="Invalid BEDS code format"),
    dict(rule_id="R011", category="Consistency",  severity="Warning", description="Graduation status without matching exit record"),
    dict(rule_id="R012", category="Timeliness",   severity="Warning", description="Record submitted after collection window closed"),
]
validation_rules = pd.DataFrame(RULE_CATALOG)
validation_rules.to_csv(DATA_DIR / "validation_rules.csv", index=False)

# Violations aggregated at district x rule x school-year (Poisson counts, scaled by (1-quality))
viol_rows = []
rule_base_rate = {  # relative frequency weight per rule (some rules fire more often than others)
    "R001": 6.0, "R002": 1.2, "R003": 2.0, "R004": 0.8, "R005": 3.5,
    "R006": 0.3, "R007": 1.5, "R008": 2.5, "R009": 0.6, "R010": 0.9,
    "R011": 1.8, "R012": 4.0,
}
for _, d in districts.iterrows():
    inv_q = 1 - d.quality_latent
    for yr in years:
        for rule in RULE_CATALOG:
            lam = max(0.02, inv_q * rule_base_rate[rule["rule_id"]] * RNG.uniform(0.6, 1.4))
            count = int(RNG.poisson(lam=lam * 8))  # scale up to plausible annual counts
            viol_rows.append(dict(
                district_id=d.district_id, region=d.region, school_year=yr,
                rule_id=rule["rule_id"], violation_count=count,
            ))

validation_violations = pd.DataFrame(viol_rows)
validation_violations = validation_violations.merge(validation_rules, on="rule_id")
validation_violations.to_csv(DATA_DIR / "validation_violations.csv", index=False)
print(f"Rule catalog: {len(validation_rules)} rules. Violation records: {len(validation_violations):,}")
validation_violations.groupby("rule_id").violation_count.sum().sort_values(ascending=False)

Rule catalog: 12 rules. Violation records: 1,440


rule_id
R001    1436
R012     870
R005     745
R008     595
R003     449
R011     383
R007     345
R002     247
R010     196
R004     167
R009     133
R006      78
Name: violation_count, dtype: int64

## 6. Synthetic student roster with injected near-duplicates

Used in Notebook 03 to demonstrate fuzzy-matching / probabilistic record-linkage duplicate detection. `is_injected_duplicate` and `duplicate_of` are **ground truth** kept only for evaluating the matching algorithm — a real system would not have this label in advance.

In [9]:
FIRST_NAMES = ["Olivia","Liam","Emma","Noah","Ava","Ethan","Sophia","Mason","Isabella","Logan",
               "Mia","Lucas","Amelia","James","Harper","Benjamin","Evelyn","Elijah","Abigail","Jacob",
               "Emily","Michael","Ella","Alexander","Scarlett","Daniel","Grace","Matthew","Chloe","Henry",
               "Layla","Jackson","Riley","Sebastian","Zoey","David","Nora","Carter","Lily","Wyatt"]
LAST_NAMES = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez",
              "Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin",
              "Lee","Perez","Thompson","White","Harris","Sanchez","Clark","Ramirez","Lewis","Robinson"]

N_STUDENTS = 3000
district_weights = (districts.enrollment_total / districts.enrollment_total.sum()).values

student_ids = RNG.choice(np.arange(900000000, 999999999), size=N_STUDENTS, replace=False)
roster_rows = []
for i in range(N_STUDENTS):
    d = districts.iloc[RNG.choice(len(districts), p=district_weights)]
    fn, ln = RNG.choice(FIRST_NAMES), RNG.choice(LAST_NAMES)
    grade = int(RNG.integers(0, 13))
    birth_year = 2026 - grade - RNG.integers(5, 7)
    dob = date(birth_year, int(RNG.integers(1, 13)), int(RNG.integers(1, 28)))
    roster_rows.append(dict(
        student_id=str(student_ids[i]),
        first_name=fn, last_name=ln, dob=str(dob),
        district_id=d.district_id, grade=grade,
        is_injected_duplicate=False, duplicate_of=None,
    ))

roster = pd.DataFrame(roster_rows)

# Inject ~150 near-duplicate records: small typos / nickname swaps / transposed DOB digits
NICKNAME_MAP = {"Olivia":"Liv","Liam":"Leo","Emma":"Em","Noah":"Noe","Ava":"Avah","Ethan":"Eathan",
                "Sophia":"Sofia","Mason":"Masen","Isabella":"Izabella","Logan":"Logen",
                "James":"Jaymes","Michael":"Micheal","Benjamin":"Benjamen","Elijah":"Elija"}

def typo(s):
    if len(s) < 4:
        return s
    i = RNG.integers(1, len(s) - 1)
    return s[:i] + s[i+1] + s[i] + s[i+2:]

n_dupes = 150
dupe_rows = []
source_idx = RNG.choice(len(roster), size=n_dupes, replace=False)
new_id_pool = RNG.choice(np.arange(700000000, 799999999), size=n_dupes, replace=False)
for k, idx in enumerate(source_idx):
    src = roster.iloc[idx]
    mode = RNG.choice(["nickname", "typo_last", "typo_first", "dob_swap"])
    fn, ln, dob_str = src.first_name, src.last_name, src.dob
    if mode == "nickname" and fn in NICKNAME_MAP:
        fn = NICKNAME_MAP[fn]
    elif mode == "typo_last":
        ln = typo(ln)
    elif mode == "typo_first":
        fn = typo(fn)
    elif mode == "dob_swap":
        y, m, dday = dob_str.split("-")
        if int(dday) <= 12:
            dob_str = f"{y}-{dday}-{m}"  # month/day transposition typo
    dupe_rows.append(dict(
        student_id=str(new_id_pool[k]),
        first_name=fn, last_name=ln, dob=dob_str,
        district_id=src.district_id, grade=src.grade,
        is_injected_duplicate=True, duplicate_of=src.student_id,
    ))

roster = pd.concat([roster, pd.DataFrame(dupe_rows)], ignore_index=True)
roster = roster.sample(frac=1, random_state=207).reset_index(drop=True)  # shuffle
roster.to_csv(DATA_DIR / "student_roster.csv", index=False)
print(f"Roster size: {len(roster):,} (including {n_dupes} injected near-duplicates)")
roster.head()

Roster size: 3,150 (including 150 injected near-duplicates)


,student_id,first_name,last_name,dob,district_id,grade,is_injected_duplicate,duplicate_of
0,977265616,James,Williams,2013-03-09,D100036,8,False,None
1,921922851,Alexander,Taylor,2010-04-16,D100011,10,False,None
2,985655366,Matthew,Johnson,2013-06-09,D100004,8,False,None
3,997721216,David,Ramirez,2018-01-22,D100039,2,False,None
4,983124935,Evelyn,Jones,2017-11-20,D100007,3,False,None


## 7. Sanity checks

In [10]:
assert districts.district_id.is_unique
assert set(submissions.district_id) <= set(districts.district_id)
assert set(idea_indicators.district_id) <= set(districts.district_id)
assert set(risk_scores.district_id) <= set(districts.district_id)
assert (submissions.completeness_rate.between(0, 1)).all()
assert (submissions.validity_error_rate.between(0, 1)).all()
assert roster.student_id.is_unique

print("All sanity checks passed.")
print()
print("Files written to ../data/:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {f.name:30s} {f.stat().st_size/1024:8.1f} KB")

All sanity checks passed.

Files written to ../data/:
  districts.csv                       5.0 KB
  idea_indicators.csv                68.1 KB
  quality_summary.csv                 0.9 KB
  risk_scores.csv                    18.5 KB
  student_roster.csv                161.3 KB
  submissions.csv                  1157.7 KB
  validation_rules.csv                0.7 KB
  validation_violations.csv         126.1 KB


## 8. Shared site-navigation config

A small config file listing the dashboard pages, used by Notebooks 02-05 to render a consistent navigation bar across every page of the site.

In [11]:
import json
CONFIG_DIR = Path("../config")
CONFIG_DIR.mkdir(exist_ok=True)

NAV_PAGES = [
    ["index.html", "Home", "../index.html"],
    ["executive-summary.html", "Executive Summary", "executive-summary.html"],
    ["data-quality.html", "Data Quality", "data-quality.html"],
    ["validation.html", "Validation & Duplicates", "validation.html"],
    ["risk-assessment.html", "Risk Assessment", "risk-assessment.html"],
    ["district-map.html", "District Map", "district-map.html"],
    ["alerts.html", "Alerts", "alerts.html"],
    ["methodology.html", "Methodology", "methodology.html"],
]
with open(CONFIG_DIR / "nav_pages.json", "w") as f:
    json.dump(NAV_PAGES, f, indent=2)
print("wrote", CONFIG_DIR / "nav_pages.json")

wrote ../config/nav_pages.json


Continue to **`02_data_quality_assessment.ipynb`** for completeness, validity, consistency, accuracy and timeliness analysis of the collections generated here.